# Indices Signal Exploration

Interactive notebook for exploring momentum signals on equity indices.

**Goals:**
1. Load and explore indices data
2. Compute momentum factors (6M returns)
3. Calculate cross-sectional ranks
4. Test different signal parameters
5. Visualize long/short signals over time

In [84]:
# Imports - PYTHON PHASE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', None)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Load Indices Data

The `lagging_indecies.py` example uses `parquet://equities/indicies.parquet`

In [85]:
# Load indices data - EXPANDED DATA LOADING
data_path = Path("../equities/indicies.parquet")

# Check if file exists
if not data_path.exists():
    print(f"⚠️ File not found at {data_path.resolve()}")
    print(f"Current working directory: {Path.cwd()}")
    print("Available files in equities/:")
    equities_path = Path("../equities")
    if equities_path.exists():
        for f in equities_path.iterdir():
            print(f"  - {f.name}")
else:
    # Load the data
    df_raw = pd.read_parquet(data_path)
    
    print(f"✅ Data loaded successfully")
    print(f"Shape: {df_raw.shape}")
    print(f"Columns: {df_raw.columns.tolist()}")
    print(f"Data types:\n{df_raw.dtypes}")
    
    # Check if there's a datetime column that needs to be set as index
    datetime_cols = df_raw.select_dtypes(include=['datetime64']).columns
    if len(datetime_cols) > 0:
        print(f"\n✅ Found datetime column: {datetime_cols[0]}")
        df_raw = df_raw.set_index(datetime_cols[0])
        print(f"Index set to: {df_raw.index.name}")
    
    # Display first few rows
    print(f"\n📊 First few rows:")
    display(df_raw.head(10))

✅ Data loaded successfully
Shape: (60370, 7)
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']
Data types:
date      datetime64[ns]
ticker            object
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object

✅ Found datetime column: date
Index set to: date

📊 First few rows:


,ticker,open,high,low,close,volume
date,,,,,,
1983-12-30,UKX,1000.0,1000.0,1000.0,1000.0,0
1984-01-03,UKX,997.5,997.5,997.5,997.5,0
1984-01-04,UKX,998.6,998.6,998.6,998.6,0
1984-01-05,UKX,1015.8,1015.8,1015.8,1015.8,0
1984-01-06,UKX,1029.0,1029.0,1029.0,1029.0,0
1984-01-09,UKX,1034.6,1034.6,1034.6,1034.6,0
1984-01-10,UKX,1034.3,1034.3,1034.3,1034.3,0
1984-01-11,UKX,1023.4,1023.4,1023.4,1023.4,0
1984-01-12,UKX,1031.3,1031.3,1031.3,1031.3,0


In [86]:
# Explore data structure - COMPREHENSIVE DATA ANALYSIS
print("="*60)
print("DATA EXPLORATION")
print("="*60)

print("\n📅 DATE RANGE:")
print(f"  From: {df_raw.index.min()}")
print(f"  To: {df_raw.index.max()}")
print(f"  Total days: {len(df_raw)}")
print(f"  Frequency: ~{len(df_raw) / ((df_raw.index.max() - df_raw.index.min()).days / 365):.1f} obs per year")

print("\n📊 INSTRUMENTS (COLUMNS):")
print(f"  {df_raw.columns.tolist()}")
print(f"  Total: {len(df_raw.columns)}")

print("\n🔍 MISSING DATA:")
missing = df_raw.isnull().sum()
if missing.sum() > 0:
    print(f"  Found missing values:")
    print(missing[missing > 0])
else:
    print(f"  ✅ No missing data")

print("\n📈 PRICE STATISTICS:")
# Select only numeric columns for describe
numeric_df = df_raw.select_dtypes(include=[np.number])
if len(numeric_df.columns) > 0:
    display(numeric_df.describe())
else:
    print("  No numeric columns found")

print("\n🔗 CORRELATION:")
if len(numeric_df.columns) > 1:
    corr = numeric_df.corr()
    display(corr)
else:
    print("  Need at least 2 numeric columns for correlation")

DATA EXPLORATION

📅 DATE RANGE:
  From: 1983-12-30 00:00:00
  To: 2025-12-12 00:00:00
  Total days: 60370
  Frequency: ~1438.0 obs per year

📊 INSTRUMENTS (COLUMNS):
  ['ticker', 'open', 'high', 'low', 'close', 'volume']
  Total: 6

🔍 MISSING DATA:
  ✅ No missing data

📈 PRICE STATISTICS:


,open,high,low,close,volume
count,60370.000000,60370.000000,60370.000000,60370.000000,6.037000e+04
mean,7970.022375,8018.557988,7916.093603,7969.020073,4.116153e+08
std,8465.870692,8499.333951,8426.692137,8464.027765,4.815778e+08
min,147.820000,149.280000,147.260000,147.820000,0.000000e+00
25%,2381.017500,2394.562500,2363.385000,2378.775000,4.496163e+07
50%,5379.620000,5416.515000,5344.605000,5382.330000,2.014988e+08
75%,9471.630000,9540.130000,9400.397500,9470.042500,6.428947e+08
max,50109.000000,50109.000000,50109.000000,50109.000000,4.463321e+09



🔗 CORRELATION:


,open,high,low,close,volume
open,1.000000,0.999962,0.999945,0.999919,0.005841
high,0.999962,1.000000,0.999914,0.999950,0.006855
low,0.999945,0.999914,1.000000,0.999959,0.004520
close,0.999919,0.999950,0.999959,1.000000,0.005644
volume,0.005841,0.006855,0.004520,0.005644,1.000000


## 2. Compute Returns & Momentum

Calculate daily returns and 6-month momentum factors

In [87]:
# Calculate returns and momentum grouped by ticker
print("="*60)
print("RETURNS & MOMENTUM CALCULATION (ALL TICKERS)")
print("="*60)

# Sort by ticker and date
df_raw_sorted = df_raw.sort_index()

# Calculate returns per ticker
def calculate_returns_momentum(ticker_data):
    """Calculate returns and momentum for a single ticker"""
    ticker_data = ticker_data.sort_index()
    
    # Daily returns
    daily_ret = ticker_data['close'].pct_change()
    
    # 6-month momentum (126 trading days)
    momentum_6m = ticker_data['close'].pct_change(periods=126)
    
    return pd.DataFrame({
        'close': ticker_data['close'],
        'daily_ret': daily_ret,
        'momentum_6m': momentum_6m
    })

# Apply function to each ticker
returns_by_ticker = {}
for ticker in sorted(df_raw['ticker'].unique()):
    ticker_data = df_raw[df_raw['ticker'] == ticker]
    returns_by_ticker[ticker] = calculate_returns_momentum(ticker_data)

print(f"\n📊 Returns calculated for {len(returns_by_ticker)} tickers:")
for ticker, ret_df in returns_by_ticker.items():
    print(f"  {ticker}: {len(ret_df)} records, returns mean={ret_df['daily_ret'].mean()*100:.4f}%")

# Store for later use
print("\n✅ Returns and momentum calculated for all tickers")

RETURNS & MOMENTUM CALCULATION (ALL TICKERS)

📊 Returns calculated for 7 tickers:
  CAC: 9251 records, returns mean=0.0252%
  CCMP: 7776 records, returns mean=0.0558%
  DAX: 7305 records, returns mean=0.0375%
  IBEX: 7797 records, returns mean=0.0323%
  MIB: 7098 records, returns mean=0.0193%
  SPX: 10525 records, returns mean=0.0425%
  UKX: 10618 records, returns mean=0.0271%

✅ Returns and momentum calculated for all tickers


## 3. Signal Performance Analysis by Ticker

Select a ticker to visualize returns and momentum signals

In [88]:
# Ticker selector and analysis with synchronized charts
from ipywidgets import widgets, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("="*60)
print("TICKER SELECTION & ANALYSIS")
print("="*60)

# Create dropdown for ticker selection
available_tickers = sorted(df_raw['ticker'].unique())
ticker_dropdown = widgets.Dropdown(
    options=available_tickers,
    value=available_tickers[0],
    description='Select Ticker:',
    style={'description_width': '120px'}
)

print(f"\n📊 Available tickers: {available_tickers}")
print("\n✅ Select a ticker from the dropdown below:")
display(ticker_dropdown)

# Output area for charts
output_area = Output()
display(output_area)

# Color scheme - matches codebase dark theme
THEME_COLORS = {
    'bg': '#0b1220',
    'panel': '#0f1b33',
    'border': '#1f2d4d',
    'text': '#e6edf7',
    'muted': '#a9b7d0',
    'grid': 'rgba(31, 45, 77, 0.3)',
    'spike': 'rgba(119, 184, 255, 0.5)',
    'price': '#1f77b4',
    'positive': '#2ca02c',
    'negative': '#d62728',
    'momentum': '#9467bd',
}

def create_returns_momentum_chart(ticker_ret, ticker_name):
    """Create synchronized 3-subplot chart: price, returns, momentum"""
    daily_returns = ticker_ret['daily_ret'].copy()
    momentum_values = ticker_ret['momentum_6m'].copy()
    
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(f"Price - {ticker_name}", "Daily Returns (%)", "6-Month Momentum"),
        specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
        vertical_spacing=0.12,
        row_heights=[0.35, 0.35, 0.3]
    )
    
    # Row 1: Price
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=ticker_ret['close'],
            name='Close Price', mode='lines',
            line=dict(color=THEME_COLORS['price'], width=2),
            fill='tozeroy', fillcolor='rgba(31, 119, 180, 0.1)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>'
        ), row=1, col=1
    )
    
    # Row 2: Returns
    colors = [THEME_COLORS['positive'] if x > 0 else THEME_COLORS['negative'] for x in daily_returns]
    fig.add_trace(
        go.Bar(
            x=ticker_ret.index, y=daily_returns * 100,
            name='Daily Return', marker=dict(color=colors), showlegend=False,
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.3f}%<extra></extra>'
        ), row=2, col=1
    )
    
    # Row 3: Momentum
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=momentum_values,
            name='6M Momentum', mode='lines',
            line=dict(color=THEME_COLORS['momentum'], width=2),
            fill='tozeroy', fillcolor='rgba(148, 103, 189, 0.2)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Momentum: %{y:.4f}<extra></extra>'
        ), row=3, col=1
    )
    
    fig.add_hline(y=0, line_dash="dash", line_color="rgba(0,0,0,0.3)", row=3, col=1)
    
    fig.update_xaxes(title_text="Date", row=3, col=1)
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Return (%)", row=2, col=1)
    fig.update_yaxes(title_text="Momentum", row=3, col=1)
    
    fig.update_layout(
        height=900,
        title_text=f"<b>Returns & Momentum Analysis - {ticker_name}</b>",
        hovermode='x unified',
        plot_bgcolor=THEME_COLORS['panel'],
        paper_bgcolor=THEME_COLORS['bg'],
        font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
        margin=dict(l=70, r=30, t=80, b=60),
        showlegend=False,
        title_font=dict(size=14, color=THEME_COLORS['text'])
    )
    
    fig.update_xaxes(
        gridcolor=THEME_COLORS['grid'],
        tickfont=dict(color=THEME_COLORS['muted']),
        showspikes=True, spikemode='across', spikethickness=1.5,
        spikecolor=THEME_COLORS['spike']
    )
    fig.update_yaxes(
        gridcolor=THEME_COLORS['grid'],
        tickfont=dict(color=THEME_COLORS['muted']),
        showspikes=True, spikethickness=1.5,
        spikecolor=THEME_COLORS['spike']
    )
    
    return fig

def update_chart(change):
    """Update chart when dropdown changes"""
    with output_area:
        output_area.clear_output(wait=True)
        selected_ticker = ticker_dropdown.value
        
        ticker_ret = returns_by_ticker[selected_ticker].copy()
        ticker_ret['momentum_6m'] = ticker_ret['close'].pct_change(periods=126)
        
        print(f"\n📊 Analyzing: {selected_ticker}")
        print(f"  Records: {len(ticker_ret)}")
        print(f"  Date range: {ticker_ret.index.min().date()} to {ticker_ret.index.max().date()}")
        
        daily_returns = ticker_ret['daily_ret'].copy()
        print(f"\n📈 Daily Returns:")
        print(f"  Mean: {daily_returns.mean()*100:.4f}%")
        print(f"  Std Dev: {daily_returns.std()*100:.4f}%")
        print(f"  Sharpe (annualized): {daily_returns.mean()/daily_returns.std() * np.sqrt(252):.3f}")
        
        momentum_values = ticker_ret['momentum_6m'].copy()
        print(f"\n💹 6-Month Momentum:")
        print(f"  Mean: {momentum_values.mean():.4f}")
        print(f"  Std Dev: {momentum_values.std():.4f}")
        
        fig = create_returns_momentum_chart(ticker_ret, selected_ticker)
        print("\n✅ Displaying synchronized analysis charts")
        fig.show()

ticker_dropdown.observe(update_chart, names='value')
update_chart(None)

TICKER SELECTION & ANALYSIS

📊 Available tickers: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

✅ Select a ticker from the dropdown below:


Dropdown(description='Select Ticker:', options=('CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX'), style=Desc…

Output()

## 4. Signal Generation & Cross-Sectional Ranking

Generate momentum signals at multiple horizons and rank tickers by signal strength

In [89]:
# Calculate momentum signals at multiple horizons
print("="*60)
print("SIGNAL GENERATION - MULTIPLE HORIZONS")
print("="*60)

# Momentum windows (trading days)
momentum_windows = {
    'mom_1m': 21,      # 1 month
    'mom_3m': 63,      # 3 months (126 trading days = 6 months)
    'mom_6m': 126,     # 6 months
    'mom_9m': 189,     # 9 months
    'mom_12m': 252,    # 12 months (1 year)
}

# Calculate signals for all tickers
signals_by_ticker = {}
for ticker, ticker_ret in returns_by_ticker.items():
    signals = ticker_ret[['close']].copy()
    
    # Calculate momentum for each window
    for signal_name, window in momentum_windows.items():
        signals[signal_name] = ticker_ret['close'].pct_change(periods=window)
    
    signals_by_ticker[ticker] = signals

print(f"\n📊 Signals calculated for {len(signals_by_ticker)} tickers")
print(f"📈 Momentum horizons: {list(momentum_windows.keys())}")
print(f"✅ Signals ready for cross-sectional ranking")

# Display sample signals for first ticker
first_ticker = list(signals_by_ticker.keys())[0]
print(f"\n🔍 Sample signals ({first_ticker}):")
display(signals_by_ticker[first_ticker].iloc[-5:])


SIGNAL GENERATION - MULTIPLE HORIZONS

📊 Signals calculated for 7 tickers
📈 Momentum horizons: ['mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m']
✅ Signals ready for cross-sectional ranking

🔍 Sample signals (CAC):


,close,mom_1m,mom_3m,mom_6m,mom_9m,mom_12m
date,,,,,,
2025-12-08,8108.43,0.019905,0.044723,0.055142,0.021443,0.092280
2025-12-09,8052.51,-0.000372,0.029269,0.040075,0.003018,0.085106
2025-12-10,8022.69,-0.016373,0.025232,0.044114,-0.006353,0.082747
2025-12-11,8085.76,-0.018866,0.023912,0.056117,-0.003550,0.099045
2025-12-12,8068.62,-0.019905,0.032028,0.068203,-0.012586,0.095432


In [90]:
# Cross-sectional ranking of signals
print("="*60)
print("CROSS-SECTIONAL RANKING")
print("="*60)

# Combine all signals into one DataFrame with hierarchical index (date, ticker)
all_signals_list = []
for ticker, signals_df in signals_by_ticker.items():
    signals_df = signals_df.copy()
    signals_df['ticker'] = ticker
    all_signals_list.append(signals_df)

all_signals = pd.concat(all_signals_list)
all_signals = all_signals.reset_index()
all_signals.columns = ['date', 'close', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m', 'ticker']
all_signals = all_signals.set_index(['date', 'ticker']).sort_index()

print(f"\n📊 Combined signals shape: {all_signals.shape}")
print(f"   Dates: {all_signals.index.get_level_values('date').min()} to {all_signals.index.get_level_values('date').max()}")
print(f"   Tickers: {sorted(all_signals.index.get_level_values('ticker').unique())}")

# Calculate cross-sectional ranks (percentile ranks)
# Rank on each date independently - higher rank = stronger signal
def rank_signals(signal_df, signal_col):
    """Rank tickers on a given date by signal strength (higher = stronger)"""
    return signal_df[signal_col].rank(pct=True, method='average')

ranks_by_date_signal = {}
for signal_col in ['mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m']:
    ranks = all_signals.groupby(level='date').apply(lambda x: rank_signals(x, signal_col))
    ranks.name = f'{signal_col}_rank'
    ranks_by_date_signal[signal_col] = ranks

# Add ranks to signals DataFrame
for signal_col, ranks in ranks_by_date_signal.items():
    all_signals[f'{signal_col}_rank'] = ranks.values

print(f"\n✅ Cross-sectional rankings calculated")

# Show ranking distribution for latest date
latest_date = all_signals.index.get_level_values('date').max()
latest_rankings = all_signals.loc[latest_date, [col for col in all_signals.columns if 'rank' in col]]

print(f"\n📈 Latest rankings ({latest_date.date()}):")
display(latest_rankings)


CROSS-SECTIONAL RANKING

📊 Combined signals shape: (60370, 6)
   Dates: 1983-12-30 00:00:00 to 2025-12-12 00:00:00
   Tickers: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

✅ Cross-sectional rankings calculated

📈 Latest rankings (2025-12-12):


,mom_1m_rank,mom_3m_rank,mom_6m_rank,mom_9m_rank,mom_12m_rank
ticker,,,,,
CAC,0.285714,0.285714,0.285714,0.142857,0.142857
CCMP,0.571429,0.714286,0.857143,1.000000,0.571429
DAX,0.857143,0.571429,0.142857,0.285714,0.714286
IBEX,1.000000,1.000000,1.000000,0.857143,1.000000
MIB,0.142857,0.142857,0.571429,0.428571,0.857143
SPX,0.714286,0.428571,0.714286,0.714286,0.285714
UKX,0.428571,0.857143,0.428571,0.571429,0.428571


In [91]:
# Visualize signal rankings heatmap
print("="*60)
print("SIGNAL RANKINGS HEATMAP")
print("="*60)

# Prepare ranking data for heatmap (last 5 years)
five_years_ago = all_signals.index.get_level_values('date').max() - pd.Timedelta(days=252*5)
recent_signals = all_signals[all_signals.index.get_level_values('date') >= five_years_ago]

# Reshape rankings for heatmap: rows = dates, columns = tickers, values = 6m momentum rank
mom_6m_rankings = recent_signals['mom_6m_rank'].unstack(fill_value=np.nan)

# Create heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
    z=mom_6m_rankings.values,
    x=mom_6m_rankings.columns,
    y=mom_6m_rankings.index,
    colorscale='RdYlGn',
    zmid=0.5,
    colorbar=dict(title="Rank<br>(Percentile)")
))

fig_heatmap.update_layout(
    title='<b>6-Month Momentum Rankings - Last 5 Years</b>',
    xaxis_title='Ticker',
    yaxis_title='Date',
    height=600,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=10, color=THEME_COLORS['text']),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_heatmap.update_xaxes(tickfont=dict(color=THEME_COLORS['muted']))
fig_heatmap.update_yaxes(tickfont=dict(color=THEME_COLORS['muted']))

print("✅ Displaying momentum rankings heatmap")
fig_heatmap.show()

print(f"\n📊 Ranking statistics (6-month momentum):")
print(f"   Mean rank: {mom_6m_rankings.mean().mean():.3f}")
print(f"   Std dev: {mom_6m_rankings.std().std():.3f}")


SIGNAL RANKINGS HEATMAP
✅ Displaying momentum rankings heatmap



📊 Ranking statistics (6-month momentum):
   Mean rank: 0.573
   Std dev: 0.047


In [103]:
# Long/Short portfolio construction based on ranks
print("="*60)
print("LONG/SHORT PORTFOLIO CONSTRUCTION")
print("="*60)

# Portfolio size for position value calculations
PORTFOLIO_SIZE = 500_000

# Create quantile-based long/short signals
# Top 50% by rank = Long (rank > 0.5)
# Bottom 50% by rank = Short (rank <= 0.5)

def create_portfolio_signals(rankings_df, quantile_threshold=0.5):
    """Create long/short portfolio signals based on ranking quantiles"""
    portfolio_signal = rankings_df.copy()
    portfolio_signal[:] = 0
    portfolio_signal[rankings_df > quantile_threshold] = 1.0   # Long
    portfolio_signal[rankings_df <= quantile_threshold] = -1.0  # Short
    return portfolio_signal

# Create long/short signals for 6-month momentum
ls_signals = create_portfolio_signals(mom_6m_rankings, quantile_threshold=0.5)

print(f"\n📊 Long/Short signal construction:")
print(f"   Quantile threshold: 0.5 (top 50% vs bottom 50%)")
print(f"   Long signal: 1.0, Short signal: -1.0")
print(f"   Portfolio size: ${PORTFOLIO_SIZE:,}")
print(f"   Position allocation: 50% to longs, 50% to shorts")

# Calculate portfolio returns for each date
ls_returns = []
dates = []
all_dates = sorted(ls_signals.index.tolist())

for i in range(len(all_dates) - 1):
    curr_date = all_dates[i]
    next_date = all_dates[i + 1]
    
    try:
        # Get signals for current date
        signal_row = ls_signals.loc[curr_date]
        long_tickers = signal_row[signal_row == 1.0].index.tolist()
        short_tickers = signal_row[signal_row == -1.0].index.tolist()
        
        if not (long_tickers or short_tickers):
            continue
        
        # Calculate position values (actual dollars deployed per position)
        long_count = len(long_tickers)
        short_count = len(short_tickers)
        
        long_position_value = (PORTFOLIO_SIZE * 0.50) / long_count if long_count > 0 else 0
        short_position_value = (PORTFOLIO_SIZE * 0.50) / short_count if short_count > 0 else 0
        
        # Accumulate portfolio P&L in dollars
        portfolio_pnl_dollars = 0.0
        
        # Process long positions (profit when underlying goes UP)
        for ticker in long_tickers:
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if not np.isnan(ret):
                    ticker_pnl = long_position_value * ret
                    portfolio_pnl_dollars += ticker_pnl
            except:
                pass
        
        # Process short positions (profit when underlying goes DOWN, so negate return)
        for ticker in short_tickers:
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if not np.isnan(ret):
                    ticker_pnl = short_position_value * (-ret)  # Negate: short profits on down moves
                    portfolio_pnl_dollars += ticker_pnl
            except:
                pass
        
        # Convert dollar P&L to portfolio return (as percentage of PORTFOLIO_SIZE)
        if portfolio_pnl_dollars != 0:
            ls_pnl = portfolio_pnl_dollars / PORTFOLIO_SIZE
            ls_returns.append(ls_pnl)
            dates.append(curr_date)
    except:
        pass

# Create performance series
ls_performance = pd.Series(ls_returns, index=dates)
ls_cumulative = (1 + ls_performance).cumprod() - 1

print(f"\n✅ Long/Short forward returns calculated (POSITION-WEIGHTED)")
print(f"   Observations: {len(ls_performance)}")
if len(ls_performance) > 0:
    print(f"   Mean daily return: {ls_performance.mean()*100:.4f}%")
    print(f"   Std dev: {ls_performance.std()*100:.4f}%")
    print(f"   Sharpe ratio (annualized): {ls_performance.mean()/ls_performance.std() * np.sqrt(252):.3f}")
    print(f"   Cumulative return: {ls_cumulative.iloc[-1]*100:.2f}%")

# Plot cumulative L-S returns
fig_ls = go.Figure()

fig_ls.add_trace(go.Scatter(
    x=ls_cumulative.index,
    y=ls_cumulative.values * 100,
    name='L-S Cumulative Return',
    mode='lines',
    line=dict(color='#22d3ee', width=2),
    fill='tozeroy',
    fillcolor='rgba(34, 211, 238, 0.1)',
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.2f}%<extra></extra>'
))

fig_ls.update_layout(
    title='<b>Long/Short Portfolio Cumulative Return (Position-Weighted by Dollar Value)</b>',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    height=500,
    hovermode='x unified',
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=70, r=30, t=80, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_ls.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)
fig_ls.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)

print("\n✅ Displaying Long/Short cumulative performance")
fig_ls.show()

LONG/SHORT PORTFOLIO CONSTRUCTION

📊 Long/Short signal construction:
   Quantile threshold: 0.5 (top 50% vs bottom 50%)
   Long signal: 1.0, Short signal: -1.0
   Portfolio size: $500,000
   Position allocation: 50% to longs, 50% to shorts

✅ Long/Short forward returns calculated (POSITION-WEIGHTED)
   Observations: 892
   Mean daily return: 0.0025%
   Std dev: 0.3438%
   Sharpe ratio (annualized): 0.113
   Cumulative return: 1.67%

✅ Displaying Long/Short cumulative performance


In [104]:
# Position tracking per ticker and performance decomposition
print("="*60)
print("POSITION TRACKING & PER-TICKER PERFORMANCE")
print("="*60)

# Track positions for each ticker over time
positions_by_ticker = {}
ticker_returns = {}
ticker_pnl = {}

for ticker in available_tickers:
    positions_by_ticker[ticker] = []
    ticker_returns[ticker] = []
    ticker_pnl[ticker] = []

# Track dates
all_backtest_dates = []

# Loop through all dates
all_unique_dates = sorted(ls_signals.index.tolist())

for i in range(len(all_unique_dates) - 1):
    curr_date = all_unique_dates[i]
    next_date = all_unique_dates[i + 1]
    
    try:
        # Get signals for current date
        signal_row = ls_signals.loc[curr_date]
        
        # Track positions and returns for each ticker
        for ticker in available_tickers:
            position = signal_row.get(ticker, 0)  # 1.0 = long, -1.0 = short, 0 = neutral
            positions_by_ticker[ticker].append(position)
            
            # Get next-day return for this ticker
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if np.isnan(ret):
                    ret = 0
            except:
                ret = 0
            
            ticker_returns[ticker].append(ret)
            
            # Calculate PnL for this position
            # PnL = position * return (long position benefits from positive returns, short benefits from negative)
            pnl = position * ret
            ticker_pnl[ticker].append(pnl)
        
        all_backtest_dates.append(curr_date)
    except Exception as e:
        pass

print(f"\n✅ Position tracking completed")
print(f"   Backtest dates: {len(all_backtest_dates)} observations")
print(f"   Tickers tracked: {available_tickers}")

# Convert to DataFrames for analysis
positions_df = pd.DataFrame(positions_by_ticker, index=all_backtest_dates)
returns_df = pd.DataFrame(ticker_returns, index=all_backtest_dates)
pnl_df = pd.DataFrame(ticker_pnl, index=all_backtest_dates)

# Calculate cumulative PnL per ticker
cumulative_pnl_by_ticker = {}
for ticker in available_tickers:
    cumulative_pnl_by_ticker[ticker] = (1 + pnl_df[ticker]).cumprod() - 1

cumulative_pnl_df = pd.DataFrame(cumulative_pnl_by_ticker, index=all_backtest_dates)

# Summary statistics per ticker
print(f"\n📊 PER-TICKER PERFORMANCE SUMMARY:")
print("-" * 60)
perf_summary = []
for ticker in available_tickers:
    mean_ret = pnl_df[ticker].mean()
    std_ret = pnl_df[ticker].std()
    sharpe = mean_ret / std_ret * np.sqrt(252) if std_ret > 0 else 0
    cumret = cumulative_pnl_df[ticker].iloc[-1]
    
    perf_summary.append({
        'Ticker': ticker,
        'Mean Daily PnL': f"{mean_ret*100:.4f}%",
        'Std Dev': f"{std_ret*100:.4f}%",
        'Sharpe': f"{sharpe:.3f}",
        'Cum Return': f"{cumret*100:.2f}%",
        'Avg Position': f"{positions_df[ticker].mean():.2f}",
        'Long Days': int((positions_df[ticker] > 0).sum()),
        'Short Days': int((positions_df[ticker] < 0).sum()),
    })

perf_summary_df = pd.DataFrame(perf_summary)
display(perf_summary_df)

print(f"\n✅ Per-ticker analysis ready")


POSITION TRACKING & PER-TICKER PERFORMANCE

✅ Position tracking completed
   Backtest dates: 892 observations
   Tickers tracked: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

📊 PER-TICKER PERFORMANCE SUMMARY:
------------------------------------------------------------


,Ticker,Mean Daily PnL,Std Dev,Sharpe,Cum Return,Avg Position,Long Days,Short Days
0,CAC,0.0076%,0.9194%,0.132,3.10%,-0.39,268,616
1,CCMP,-0.0015%,1.3428%,-0.017,-9.00%,0.26,549,317
2,DAX,0.0461%,0.9559%,0.766,44.86%,0.33,590,292
3,IBEX,0.0098%,0.9267%,0.168,5.03%,0.55,689,195
4,MIB,0.0307%,1.0437%,0.466,25.21%,0.45,638,240
5,SPX,0.0173%,1.0197%,0.270,11.39%,0.20,523,343
6,UKX,-0.0243%,0.6968%,-0.553,-21.19%,-0.43,244,629



✅ Per-ticker analysis ready


In [97]:
# Per-ticker signals and performance with interactive toggle
print("="*60)
print("PER-TICKER SIGNALS & PERFORMANCE ANALYSIS")
print("="*60)

from ipywidgets import widgets, HBox, VBox

# INTERACTIVE SIGNALS & PnL CHARTS
print("\n✅ Creating interactive signal and performance analysis...\n")

# Create checkbox for each ticker
ticker_checkboxes = {}
for ticker in available_tickers:
    ticker_checkboxes[ticker] = widgets.Checkbox(
        value=True,
        description=ticker,
        indent=False
    )

# Button to toggle all on/off
toggle_all_button = widgets.Button(description='Toggle All')
toggle_all_state = {'value': True}

def toggle_all_clicked(b):
    toggle_all_state['value'] = not toggle_all_state['value']
    for ticker in available_tickers:
        ticker_checkboxes[ticker].value = toggle_all_state['value']

toggle_all_button.on_click(toggle_all_clicked)

# Output area for chart
perf_output_area = widgets.Output()

def update_perf_chart(change=None):
    """Update performance chart based on checkbox selections"""
    with perf_output_area:
        perf_output_area.clear_output(wait=True)
        
        # Determine which tickers to show
        selected_tickers = [t for t in available_tickers if ticker_checkboxes[t].value]
        
        if not selected_tickers:
            print("⚠️ Select at least one ticker to display")
            return
        
        # Calculate position values in dollars for each date
        # Position value = signal * (50% / count) * portfolio_size
        position_values = {}
        for ticker in available_tickers:
            values = []
            for date_idx, date in enumerate(positions_df.index):
                signal = positions_df.loc[date, ticker]
                
                if signal == 0:
                    values.append(0)
                elif signal > 0:  # Long position
                    long_count = (positions_df.loc[date] > 0).sum()
                    if long_count > 0:
                        allocation = (PORTFOLIO_SIZE * 0.50) / long_count
                        values.append(allocation)  # Positive for long
                    else:
                        values.append(0)
                else:  # Short position (signal < 0)
                    short_count = (positions_df.loc[date] < 0).sum()
                    if short_count > 0:
                        allocation = (PORTFOLIO_SIZE * 0.50) / short_count
                        values.append(-allocation)  # Negative for short
                    else:
                        values.append(0)
            
            position_values[ticker] = values
        
        position_values_df = pd.DataFrame(position_values, index=positions_df.index)
        
        # Create subplots with signals on top, cumulative PnL below
        fig_perf = make_subplots(
            rows=3, cols=1,
            subplot_titles=("Trading Signal (+1.0=Long, -1.0=Short)", 
                           "Position Value in $ (+Long / -Short)", 
                           "Cumulative PnL by Ticker"),
            vertical_spacing=0.12,
            row_heights=[0.3, 0.3, 0.4]
        )
        
        # Add signal traces (top chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=positions_df.index,
                    y=positions_df[ticker],
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    hovertemplate=f'<b>{ticker}</b><br>%{{x|%Y-%m-%d}}<br>Signal: %{{y:.2f}}<extra></extra>',
                    legendgroup=ticker,
                    showlegend=True
                ),
                row=1, col=1
            )
        
        # Add reference lines for signals
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=1, col=1)
        fig_perf.add_hline(y=1, line_dash="dot", line_color="rgba(44, 160, 44, 0.3)", row=1, col=1)
        fig_perf.add_hline(y=-1, line_dash="dot", line_color="rgba(214, 39, 40, 0.3)", row=1, col=1)
        
        # Add position value traces (middle chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=position_values_df.index,
                    y=position_values_df[ticker],
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    fill='tozeroy',
                    hovertemplate='<b>' + ticker + '</b><br>%{x|%Y-%m-%d}<br>Position: $%{y:,.0f}<extra></extra>',
                    legendgroup=ticker,
                    showlegend=False
                ),
                row=2, col=1
            )
        
        # Add zero line for position value
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=2, col=1)
        
        # Add cumulative PnL traces (bottom chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=cumulative_pnl_df.index,
                    y=cumulative_pnl_df[ticker] * 100,
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    hovertemplate=f'<b>{ticker}</b><br>%{{x|%Y-%m-%d}}<br>Return: %{{y:.2f}}%<extra></extra>',
                    legendgroup=ticker,
                    showlegend=False
                ),
                row=3, col=1
            )
        
        # Add zero line for PnL
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=3, col=1)
        
        # Update axes
        fig_perf.update_xaxes(title_text="Date", row=3, col=1)
        fig_perf.update_yaxes(title_text="Signal", row=1, col=1)
        fig_perf.update_yaxes(title_text="Position Value ($)", row=2, col=1)
        fig_perf.update_yaxes(title_text="Return (%)", row=3, col=1)
        
        fig_perf.update_layout(
            title='<b>Per-Ticker Signals, Position Values & Cumulative PnL</b>',
            height=1200,
            hovermode='x unified',
            plot_bgcolor=THEME_COLORS['panel'],
            paper_bgcolor=THEME_COLORS['bg'],
            font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
            margin=dict(l=100, r=30, t=80, b=60),
            title_font=dict(size=14, color=THEME_COLORS['text']),
            legend=dict(bgcolor='rgba(31, 45, 77, 0.5)', bordercolor=THEME_COLORS['border'], borderwidth=1)
        )
        
        fig_perf.update_xaxes(
            gridcolor=THEME_COLORS['grid'],
            tickfont=dict(color=THEME_COLORS['muted'])
        )
        fig_perf.update_yaxes(
            gridcolor=THEME_COLORS['grid'],
            tickfont=dict(color=THEME_COLORS['muted'])
        )
        
        fig_perf.show()

# Connect checkboxes to update function
for ticker in available_tickers:
    ticker_checkboxes[ticker].observe(update_perf_chart, names='value')

# Layout
checkbox_layout = HBox([ticker_checkboxes[t] for t in available_tickers[:4]])
checkbox_layout2 = HBox([ticker_checkboxes[t] for t in available_tickers[4:]])
controls = VBox([
    widgets.HTML("<b>Select Tickers to Display:</b>"),
    checkbox_layout,
    checkbox_layout2,
    toggle_all_button
])

display(controls)
display(perf_output_area)

# Initial display
update_perf_chart()

print("✅ Per-ticker performance analysis ready - toggle checkboxes to show/hide tickers")


PER-TICKER SIGNALS & PERFORMANCE ANALYSIS

✅ Creating interactive signal and performance analysis...



Output()

✅ Per-ticker performance analysis ready - toggle checkboxes to show/hide tickers


In [95]:
# Detailed position history and contribution analysis
print("="*60)
print("POSITION HISTORY & CONTRIBUTION ANALYSIS")
print("="*60)

# 1. Position transition table
print("\n📊 POSITION TRANSITIONS BY TICKER:")
print("-" * 80)

position_transitions = []
for ticker in available_tickers:
    long_count = (positions_df[ticker] > 0).sum()
    short_count = (positions_df[ticker] < 0).sum()
    neutral_count = (positions_df[ticker] == 0).sum()
    
    position_transitions.append({
        'Ticker': ticker,
        'Long %': f"{100*long_count/len(positions_df):.1f}%",
        'Short %': f"{100*short_count/len(positions_df):.1f}%",
        'Neutral %': f"{100*neutral_count/len(positions_df):.1f}%",
    })

trans_df = pd.DataFrame(position_transitions)
display(trans_df)

# 2. Correlation of returns
print("\n🔗 TICKER RETURN CORRELATIONS:")
print("-" * 80)
ticker_corr = returns_df.corr()
display(ticker_corr)

# 3. Contribution to overall P&L
print("\n💰 CONTRIBUTION TO OVERALL L/S P&L:")
print("-" * 80)

# Calculate cumulative contribution per ticker
contributions = []
total_pnl = pnl_df.sum().sum()

for ticker in available_tickers:
    ticker_total_pnl = pnl_df[ticker].sum()
    ticker_contribution = ticker_total_pnl / total_pnl if total_pnl != 0 else 0
    
    contributions.append({
        'Ticker': ticker,
        'Total PnL': f"${ticker_total_pnl*10000:.2f}",  # Scale for readability
        'Contribution %': f"{ticker_contribution*100:.1f}%",
        'Avg Daily PnL': f"{pnl_df[ticker].mean()*100:.4f}%",
        'Win Rate': f"{(pnl_df[ticker] > 0).sum() / len(pnl_df[ticker]) * 100:.1f}%"
    })

contrib_df = pd.DataFrame(contributions)
display(contrib_df)

# 4. Comparative performance chart
print("\n📈 Creating comparative performance dashboard...\n")

fig_compare = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Cumulative Return by Ticker", "Win Rate Distribution", 
                   "Sharpe Ratio by Ticker", "Position Exposure Over Time"),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Row 1, Col 1: Cumulative returns
for ticker in available_tickers:
    fig_compare.add_trace(
        go.Scatter(x=cumulative_pnl_df.index, y=cumulative_pnl_df[ticker] * 100,
                   name=ticker, mode='lines'),
        row=1, col=1
    )

# Row 1, Col 2: Win rate bar chart
win_rates = [(pnl_df[ticker] > 0).sum() / len(pnl_df[ticker]) * 100 for ticker in available_tickers]
fig_compare.add_trace(
    go.Bar(x=available_tickers, y=win_rates, name='Win Rate %', marker_color='rgba(100, 200, 100, 0.7)',
           hovertemplate='<b>%{x}</b><br>Win Rate: %{y:.1f}%<extra></extra>'),
    row=1, col=2
)

# Row 2, Col 1: Sharpe ratios
sharpe_ratios = []
for ticker in available_tickers:
    mean_ret = pnl_df[ticker].mean()
    std_ret = pnl_df[ticker].std()
    sharpe = mean_ret / std_ret * np.sqrt(252) if std_ret > 0 else 0
    sharpe_ratios.append(sharpe)

colors_sharpe = ['green' if s > 0 else 'red' for s in sharpe_ratios]
fig_compare.add_trace(
    go.Bar(x=available_tickers, y=sharpe_ratios, name='Sharpe Ratio', marker_color=colors_sharpe,
           hovertemplate='<b>%{x}</b><br>Sharpe: %{y:.3f}<extra></extra>'),
    row=2, col=1
)

# Row 2, Col 2: Net position exposure over time
net_exposure = (positions_df > 0).sum(axis=1) - (positions_df < 0).sum(axis=1)
fig_compare.add_trace(
    go.Scatter(x=positions_df.index, y=net_exposure, name='Net Exposure',
               mode='lines', fill='tozeroy', line=dict(color='#22d3ee', width=2),
               hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Net Longs - Shorts: %{y:.0f}<extra></extra>'),
    row=2, col=2
)

# Update axes labels
fig_compare.update_xaxes(title_text="Date", row=1, col=1)
fig_compare.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig_compare.update_xaxes(title_text="Ticker", row=1, col=2)
fig_compare.update_yaxes(title_text="Win Rate (%)", row=1, col=2)
fig_compare.update_xaxes(title_text="Ticker", row=2, col=1)
fig_compare.update_yaxes(title_text="Sharpe Ratio", row=2, col=1)
fig_compare.update_xaxes(title_text="Date", row=2, col=2)
fig_compare.update_yaxes(title_text="# of Long Positions - # of Short", row=2, col=2)

# Update layout
fig_compare.update_layout(
    title='<b>Comprehensive Performance Dashboard</b>',
    height=900,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=10, color=THEME_COLORS['text']),
    title_font=dict(size=14, color=THEME_COLORS['text']),
    showlegend=True,
    hovermode='closest'
)

fig_compare.update_xaxes(gridcolor=THEME_COLORS['grid'], tickfont=dict(color=THEME_COLORS['muted']))
fig_compare.update_yaxes(gridcolor=THEME_COLORS['grid'], tickfont=dict(color=THEME_COLORS['muted']))

print("✅ Displaying comprehensive performance dashboard")
fig_compare.show()

print("\n✅ Position history and contribution analysis complete")


POSITION HISTORY & CONTRIBUTION ANALYSIS

📊 POSITION TRANSITIONS BY TICKER:
--------------------------------------------------------------------------------


,Ticker,Long %,Short %,Neutral %
0,CAC,30.0%,69.1%,0.9%
1,CCMP,61.5%,35.5%,2.9%
2,DAX,66.1%,32.7%,1.1%
3,IBEX,77.2%,21.9%,0.9%
4,MIB,71.5%,26.9%,1.6%
5,SPX,58.6%,38.5%,2.9%
6,UKX,27.4%,70.5%,2.1%



🔗 TICKER RETURN CORRELATIONS:
--------------------------------------------------------------------------------


,CAC,CCMP,DAX,IBEX,MIB,SPX,UKX
CAC,1.000000,0.332107,0.875300,0.766021,0.852961,0.375973,0.745352
CCMP,0.332107,1.000000,0.384661,0.275488,0.330441,0.963499,0.214709
DAX,0.875300,0.384661,1.000000,0.790170,0.861388,0.414354,0.716385
IBEX,0.766021,0.275488,0.790170,1.000000,0.819466,0.332699,0.722837
MIB,0.852961,0.330441,0.861388,0.819466,1.000000,0.377801,0.713771
SPX,0.375973,0.963499,0.414354,0.332699,0.377801,1.000000,0.291396
UKX,0.745352,0.214709,0.716385,0.722837,0.713771,0.291396,1.000000



💰 CONTRIBUTION TO OVERALL L/S P&L:
--------------------------------------------------------------------------------


,Ticker,Total PnL,Contribution %,Avg Daily PnL,Win Rate
0,CAC,$681.43,8.9%,0.0076%,48.9%
1,CCMP,$-131.78,-1.7%,-0.0015%,50.6%
2,DAX,$4114.11,53.7%,0.0461%,50.0%
3,IBEX,$874.61,11.4%,0.0098%,52.4%
4,MIB,$2735.67,35.7%,0.0307%,52.1%
5,SPX,$1545.50,20.2%,0.0173%,50.1%
6,UKX,$-2163.67,-28.3%,-0.0243%,47.2%



📈 Creating comparative performance dashboard...

✅ Displaying comprehensive performance dashboard



✅ Position history and contribution analysis complete


In [105]:

# Portfolio value evolution and returns analysis
print("="*80)
print("PORTFOLIO VALUE & RETURNS ANALYSIS")
print("="*80)

# Portfolio size ($500K - margin account, capital deployed: 50% long + 50% short = 100% leverage)
PORTFOLIO_SIZE = 500_000

# Calculate daily portfolio returns from L-S strategy
portfolio_returns = ls_performance.copy()

# Calculate portfolio value over time
portfolio_value = PORTFOLIO_SIZE * (1 + portfolio_returns).cumprod()

# Summary statistics
initial_value = PORTFOLIO_SIZE
final_value = portfolio_value.iloc[-1]
total_return_dollars = final_value - initial_value
total_return_pct = (final_value / initial_value - 1) * 100

# Daily statistics
mean_daily_ret = portfolio_returns.mean()
std_daily_ret = portfolio_returns.std()
sharpe_ratio = mean_daily_ret / std_daily_ret * np.sqrt(252) if std_daily_ret > 0 else 0

# Drawdown analysis
cumulative_returns = (1 + portfolio_returns).cumprod()
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max
max_drawdown = drawdown.min()

print(f"""
💰 PORTFOLIO PERFORMANCE SUMMARY (Starting with ${PORTFOLIO_SIZE:,.0f})
═════════════════════════════════════════════════════════════════════════════

FINAL RESULTS:
  Initial Portfolio Value:  ${initial_value:>15,.0f}
  Final Portfolio Value:    ${final_value:>15,.0f}
  Total Return ($):         ${total_return_dollars:>15,.0f}
  Total Return (%):         {total_return_pct:>15.2f}%

PERFORMANCE METRICS:
  Mean Daily Return:        {mean_daily_ret*100:>15.4f}%
  Daily Std Deviation:      {std_daily_ret*100:>15.4f}%
  Sharpe Ratio (annual):    {sharpe_ratio:>15.3f}
  Max Drawdown:             {max_drawdown*100:>15.2f}%
  
  Backtest Period:          {portfolio_returns.index.min().date()} to {portfolio_returns.index.max().date()}
  Trading Days:             {len(portfolio_returns):>15,.0f}

═════════════════════════════════════════════════════════════════════════════
""")

# Monthly returns table
portfolio_value_df = pd.DataFrame({
    'Portfolio_Value': portfolio_value,
    'Daily_Return': portfolio_returns,
})

# Extract month-end values
monthly_values = portfolio_value_df['Portfolio_Value'].resample('M').last()
monthly_returns = portfolio_value_df['Daily_Return'].resample('M').sum()

print("\n📅 MONTHLY RETURNS (Year-Month: Return $):")
print("-" * 80)
monthly_summary = []
for date, ret in monthly_returns.items():
    ret_dollars = monthly_values.loc[date] - monthly_values.shift(1).loc[date]
    monthly_summary.append({
        'Date': date.strftime('%Y-%m'),
        'Return $': ret_dollars,
        'Return %': ret * 100,
        'Portfolio Value': monthly_values.loc[date]
    })

monthly_df = pd.DataFrame(monthly_summary)
display(monthly_df.tail(12))  # Show last 12 months

# Create portfolio value chart
print("\n📈 Creating portfolio value chart over time...\n")

fig_portfolio = go.Figure()

# Portfolio value line
fig_portfolio.add_trace(go.Scatter(
    x=portfolio_value.index,
    y=portfolio_value.values,
    name='Portfolio Value',
    mode='lines',
    line=dict(color='#10b981', width=3),
    fill='tozeroy',
    fillcolor='rgba(16, 185, 129, 0.1)',
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Value: $%{y:,.0f}<extra></extra>'
))

# Add initial value reference line
fig_portfolio.add_hline(y=initial_value, line_dash="dash", line_color="rgba(255,255,255,0.2)", 
                       annotation_text="Initial", annotation_position="right")

fig_portfolio.update_layout(
    title=f'<b>Portfolio Value Over Time</b><br><sub>Starting Value: ${initial_value:,.0f} | Final Value: ${final_value:,.0f} | Total Return: {total_return_pct:.2f}%</sub>',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    height=600,
    hovermode='x unified',
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=100, r=30, t=100, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_portfolio.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikemode='across', spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)
fig_portfolio.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)

print("✅ Displaying portfolio value chart")
fig_portfolio.show()

# Create dual chart: cumulative return % and daily returns distribution
fig_dual = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Cumulative Return %", "Daily Returns Distribution"),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4]
)

# Top: Cumulative return %
cumulative_return_pct = ((portfolio_value / initial_value) - 1) * 100
fig_dual.add_trace(
    go.Scatter(
        x=portfolio_value.index,
        y=cumulative_return_pct.values,
        name='Cumulative Return %',
        mode='lines',
        line=dict(color='#3b82f6', width=2),
        fill='tozeroy',
        fillcolor='rgba(59, 130, 246, 0.1)',
        hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.2f}%<extra></extra>'
    ),
    row=1, col=1
)

fig_dual.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=1, col=1)

# Bottom: Daily returns distribution
fig_dual.add_trace(
    go.Histogram(
        x=portfolio_returns.values * 100,
        name='Daily Returns %',
        nbinsx=50,
        marker=dict(color='#8b5cf6'),
        hovertemplate='Return Range: %{x:.3f}%<br>Frequency: %{y}<extra></extra>'
    ),
    row=2, col=1
)

fig_dual.update_xaxes(title_text="Date", row=1, col=1)
fig_dual.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig_dual.update_xaxes(title_text="Daily Return (%)", row=2, col=1)
fig_dual.update_yaxes(title_text="Frequency", row=2, col=1)

fig_dual.update_layout(
    title='<b>Portfolio Returns Analysis</b>',
    height=800,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=70, r=30, t=80, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text']),
    showlegend=False
)

fig_dual.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)
fig_dual.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)

print("✅ Displaying returns analysis")
fig_dual.show()

print("\n✅ Portfolio value analysis complete")


PORTFOLIO VALUE & RETURNS ANALYSIS

💰 PORTFOLIO PERFORMANCE SUMMARY (Starting with $500,000)
═════════════════════════════════════════════════════════════════════════════

FINAL RESULTS:
  Initial Portfolio Value:  $        500,000
  Final Portfolio Value:    $        508,370
  Total Return ($):         $          8,370
  Total Return (%):                    1.67%

PERFORMANCE METRICS:
  Mean Daily Return:                 0.0025%
  Daily Std Deviation:               0.3438%
  Sharpe Ratio (annual):              0.113
  Max Drawdown:                       -5.49%

  Backtest Period:          2022-07-01 to 2025-12-11
  Trading Days:                         892

═════════════════════════════════════════════════════════════════════════════


📅 MONTHLY RETURNS (Year-Month: Return $):
--------------------------------------------------------------------------------


C:\Users\andre\AppData\Local\Temp\ipykernel_16088\472527526.py:61: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\andre\AppData\Local\Temp\ipykernel_16088\472527526.py:62: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,Date,Return $,Return %,Portfolio Value
30,2025-01,-7276.921662,-1.403932,510797.750093
31,2025-02,6030.584021,1.179114,516828.334114
32,2025-03,6191.756720,1.222530,523020.090834
33,2025-04,-11373.032524,-2.048081,511647.058310
34,2025-05,-716.335587,-0.128650,510930.722723
35,2025-06,-9099.736049,-1.791240,501830.986675
36,2025-07,-2331.489058,-0.461323,499499.497617
37,2025-08,4072.962247,0.816515,503572.459864
38,2025-09,433.546542,0.090581,504006.006407
39,2025-10,3993.443376,0.796205,507999.449783



📈 Creating portfolio value chart over time...

✅ Displaying portfolio value chart


✅ Displaying returns analysis



✅ Portfolio value analysis complete
